[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/46_mixup_solution.ipynb)

# ✅ Solution: Mixup

**Mixup** (Zhang et al. 2017) を実装する。サンプルペアと label を線形補間する augmentation。CIFAR-10 / ImageNet 上位レシピの常連。

$$\tilde{x} = \lambda x + (1-\lambda) x_{\text{perm}}, \quad \lambda \sim \text{Beta}(\alpha, \alpha)$$

> 💡 **どこで使う**：ペア画像と label を λ : 1-λ で線形補間する augmentation。決定境界がスムーズになり、汎化が改善。

<details>
<summary>📖 もっと詳しく</summary>

Zhang 2017。`α=0.2–1.0` の Beta 分布から λ をサンプル — `α` を上げるほど混ぜが強い (`α=1.0` で uniform)。

標準 PyTorch 流: `(x_mix, y_a, y_b, lam)` を返し、loss は外で `lam*CE(y_a) + (1-lam)*CE(y_b)`。Cutout (#45)、CutMix (#47) と組み合わせ可能。

</details>

### 4-tuple interface
標準的な PyTorch interface は両 label と mixing 係数を返し、loss は外で計算する：
```python
x_mix, y_a, y_b, lam = mixup(x, y)
loss = lam * CE(model(x_mix), y_a) + (1 - lam) * CE(model(x_mix), y_b)
```

### Signature
```python
def mixup(x, y, alpha=1.0):
    # x: (B, ...) inputs
    # y: (B,) long class indices
    # alpha: Beta distribution parameter (>0)
    # Returns: (x_mixed, y_a, y_b, lam)
```

### Rules
- `λ ~ Beta(α, α)` をサンプル — `torch.distributions.Beta(α, α).sample().item()` を使う
- `torch.randperm(B)` で batch を permute
- **4-tuple** `(x_mixed, y_a, y_b, lam)` を return
- `y_a` は元の `y`、`y_b` は `y[perm]`
- `x_mixed = lam * x + (1 - lam) * x[perm]`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def mixup(x, y, alpha=1.0):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    B = x.shape[0]
    perm = torch.randperm(B, device=x.device)
    x_mixed = lam * x + (1 - lam) * x[perm]
    return x_mixed, y, y[perm], lam

In [ ]:
import torch
torch.manual_seed(0)
x = torch.randn(4, 3, 8, 8)
y = torch.tensor([0, 1, 2, 3])
x_mix, y_a, y_b, lam = mixup(x, y, alpha=1.0)
print('x_mix shape:', x_mix.shape, '| lam =', round(lam, 3))
print('y_a:', y_a.tolist(), '| y_b:', y_b.tolist())

In [ ]:
from torch_judge import check
check("mixup")